# Nepal Border Sentiment Analysis

Fine-tuning a multilingual transformer (XLM-RoBERTa) to classify sentiment in Nepali/English YouTube comments discussing the Nepal–India border dispute and Nepal PM's related remarks.

**Pipeline:** scrape YouTube comments → translate Nepali to English → auto-label sentiment → manually verify a gold-standard subset → fine-tune XLM-RoBERTa → evaluate.

**Note on data sources tried:** Twitter/X API was dropped (paid). Nepali news site comment sections were explored using BeautifulSoup, but most had either no comments or used an embedded Facebook Comments plugin. YouTube comments ended up being the most viable source — high volume, no API key required, and directly relevant given ~70%+ of Nepali internet users are active there.

## Setup

Install dependencies.

In [1]:
!pip install -q youtube-comment-downloader
!pip install -q pandas
!pip install -q deep-translator
!pip install -q tqdm
!pip install -q transformers
!pip install -q torch
!pip install -q tiktoken
!pip install -q sentencepiece protobuf
!pip install -q datasets
!pip install -q scikit-learn
!pip install -q 'accelerate>=1.1.0'


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip insta

## 1. Data Collection

Scraping comments from 17 YouTube videos across major Nepali news channels (Kantipur, 
BBC Nepali, KhojSamachar, etc.) and independent commentary channels covering the border 
dispute, using `youtube-comment-downloader` since it requires no API key, unlike the 
YouTube Data API or Twitter API.

In [2]:
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd

In [3]:
# Initial scraping - run once, then load from CSV

# cmt_downloader = YoutubeCommentDownloader()
# urls={'https://www.youtube.com/watch?v=7XaHoexB7rQ':'KantipurTV',
#       'https://www.youtube.com/watch?v=FqyGil9SZ9g':'TankaDahal', 
#       'https://www.youtube.com/watch?v=5S-KM7o_DRk':'SagarmathaTV', 
#       'https://www.youtube.com/watch?v=ryhQNybiElQ':'TheNepaliComment',
#       'https://www.youtube.com/watch?v=U2jbDkyGWx4':'IDSMediaNetwork',
#       'https://www.youtube.com/watch?v=COGGc-UvlaI':'KhojSamachar',
#       'https://www.youtube.com/watch?v=eTb8SyZun1M':'Samacharpati',
#       'https://www.youtube.com/watch?v=c9X8FaPruKI':'BBCNewsNepali',
#       'https://www.youtube.com/watch?v=JJpkqAM-uf0':'InsideNepalNews',
#       'https://www.youtube.com/watch?v=PS7elNlcdHg':'NewsInNepal',
#       'https://www.youtube.com/watch?v=ZuMOJjCsthA':'News24Nepal',
#       'https://www.youtube.com/watch?v=YcBSKgWFGXA':'GSMediaNetwork',
#       'https://www.youtube.com/watch?v=45rp5GC0tcQ':'KhojSamachar',
#       'https://www.youtube.com/watch?v=Mf7YpiqHYis':'TechPana',
#       'https://www.youtube.com/watch?v=QZy_b9XNFfo':'TVTodayHD',
#       'https://www.youtube.com/watch?v=QeLbM6LdLBA':'WhySoOffended',
#       'https://www.youtube.com/watch?v=Kp2sVewoQWA':'Kछ Nepal'
#      }

# all_data=[]

# for url, channel in urls.items():
#     try:
#         comments = cmt_downloader.get_comments_from_url(url)
#         for comment in comments:
#             all_data.append({
#                 'comment': comment['text'],
#                 'channel': channel,
#                 'video_url': url
#             })

#             if len([d for d in all_data if d['video_url'] == url]) >= 200:
#                 break
#     except Exception as e:
#         print(f"Failed {url}: {e}")
#         continue

# df = pd.DataFrame(all_data)
# df.to_csv('nepal_border_comments.csv', index=False)
# print(f"Total comments: {len(df)}")

In [4]:
# Skip rescraping - load from existing CSV
df = pd.read_csv('nepal_border_comments.csv')
print(f"Loaded: {len(df)}")

Loaded: 2266


### Deduplication

Removing duplicate comments to prevent data leakage between training and evaluation sets. Duplicate comments across videos (popular phrases, pinned comments, generic reactions) can artificially inflate evaluation metrics if they appear in both splits.


In [5]:
# Dedupe at the source
df = df.drop_duplicates(subset=['comment']).reset_index(drop=True)
print(f"After dedup: {len(df)}")

After dedup: 2197


## 2. Translation (Nepali → English)

Most comments are in Nepali (Devanagari script), Romanized Nepali, or a mix of Nepali and English (code-switching). To make labeling and modeling more tractable, all comments are translated to English using Google Translate via `deep-translator`.

This step has known limitations — Nepali slang, sarcasm, and political idioms often translate poorly or lose nuance. ~3.5% of comments failed to translate and were dropped.

In [6]:
from deep_translator import GoogleTranslator
import time

In [7]:
def translate_text(text):
    try:
        return GoogleTranslator(source='auto',target='en').translate(str(text)[:500])
    except Exception as e:
        return None

In [8]:
from tqdm import tqdm
tqdm.pandas()

df['comment_en'] = df['comment'].progress_apply(translate_text)
df.to_csv('nepal_comments_translated.csv', index=False)
print(f"Done. Failed translations: {df['comment_en'].isna().sum()}")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2197/2197 [08:16<00:00,  4.43it/s]

Done. Failed translations: 52


In [9]:
#drop failed translations
df = pd.read_csv('nepal_comments_translated.csv')

df = df.dropna(subset=['comment_en'])
print(f"length of remaining records :: {len(df)}")

print(df[['comment', 'comment_en']].sample(10).to_string())

length of remaining records :: 2145
                                                                                                                                                                                                                                                                                                                                                                                                                       comment                                                                                                                                                                                                                                                                                                                                                                                                                comment_en
2186                                                                                                                                       

# 3. Auto-Labeling 

Using a pretrained multilingual sentiment model (`cardiffnlp/twitter-xlm-roberta-base-sentiment`) to generate an initial sentiment label for every comment. This gives a large but noisy labeled set, since the model has limited exposure to Nepali political slang, sarcasm, and code-switched text.

In [10]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    truncation=True,
    max_length=512
)

def get_sentiment(comment_text):
    try:
        result = classifier(str(comment_text[:512]))[0]['label']
        return result
    except:
        return None
    


tqdm.pandas()
df['sentiment']=df['comment_en'].progress_apply(get_sentiment)
print(df['sentiment'].value_counts())
df.to_csv('nepal_comments_labelled.csv',index=False)
    

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2145/2145 [01:34<00:00, 22.59it/s]


sentiment
negative    1130
neutral      563
positive     452
Name: count, dtype: int64


### Spot-checking auto-labels

Manually inspecting a few samples per class to gauge auto-label quality before deciding on a verification strategy. Several mislabels were found (e.g., sarcastic or slang-heavy comments mislabeled by sentiment), which motivated creating a manually verified subset
(see next section).

In [11]:
# Check 20 random samples across all three classes
for sentiment in ['positive', 'negative', 'neutral']:
    print(f"\n--- {sentiment.upper()} SAMPLES ---")
    sample_sentiment = df[df['sentiment'] == sentiment][['comment', 'comment_en', 'sentiment']].sample(3)
    print(sample_sentiment.to_string())


--- POSITIVE SAMPLES ---
                                                                          comment                                                                  comment_en sentiment
1481                           Ho sir yo ❤farmolale Bharat ma hanggama hunx thikx                           Ho sir yo ❤farmolale Bharat ma hangama hunx thikx  positive
2131  Katie ra.pra .pa \nPa navayako pra pra ...  Aru le jandaina lu Rajendra sir  Katie ra.pra .pa \nPa navayako pra pra ... Aru le jandaina lu Rajendra sir  positive
219                                                        Balen.sir.pm.jindabad.                                                      Balen.sir.pm.zindabad.  positive

--- NEGATIVE SAMPLES ---
                                                                                                                                                                                                                                                                                    

## 4. Manual Verification

Sampling 100 comments per sentiment class (300 total) and manually correcting the auto-generated labels by hand in a spreadsheet. This becomes the gold-standard evaluation set — never used in training, only for honest evaluation.

This follows the standard silver/gold-standard split used in low-resource NLP research (e.g., SNLI), where a large noisy auto-labeled set is used for training, and a smaller human-verified set is used for evaluation.

In [12]:
# Get 100 samples from each class to review
df_sample = df.groupby('sentiment').sample(n=100, replace=False).reset_index(drop=True)
print(df_sample.columns)

df_sample.to_csv('manual_review.csv', index=False)

Index(['comment', 'channel', 'video_url', 'comment_en', 'sentiment'], dtype='str')


In [13]:
df_verified = pd.read_csv('comments_manual_review.csv')
print(df_verified['sentiment'].value_counts())

sentiment
neutral     159
negative     72
positive     69
Name: count, dtype: int64


## 5. Train / Eval Split

The gold-standard 300 samples are held out entirely as the evaluation set. The remaining auto-labeled comments form the training set.

In [16]:
df_verified = pd.read_csv('manual_review.csv')

# Exclude eval comments from training by comment text (not index)
eval_comments_set = set(df_verified['comment_en'])
df_train = df[~df['comment_en'].isin(eval_comments_set)]

# Verify clean split
overlap_check = set(df_train['comment_en']).intersection(eval_comments_set)
print(f"Remaining overlap: {len(overlap_check)}")
print(f"Train size: {len(df_train)}, Eval size: {len(df_verified)}")

Remaining overlap: 0
Train size: 1845, Eval size: 300


## 6. Tokenization & Dataset Preparation

Loading the XLM-RoBERTa tokenizer and converting both the training and evaluation sets into HuggingFace `Dataset` objects.

In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

# Encode labels
le = LabelEncoder()
df_train['label'] = le.fit_transform(df_train['sentiment'])
df_verified['label'] = le.transform(df_verified['sentiment'])

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))



Label mapping: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [18]:
# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(df_train[['comment_en', 'label']].dropna())
eval_dataset = Dataset.from_pandas(df_verified[['comment_en', 'label']])

# Load tokenizer
model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize
def tokenize(batch):
    return tokenizer(batch['comment_en'], truncation=True, max_length=128, padding='max_length')

train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset = eval_dataset.map(tokenize, batched=True)

print("Dataset ready")
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

Map:   0%|          | 0/1845 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset ready
Train: 1845 | Eval: 300


## 7. Fine-Tuning XLM-RoBERTa

Fine-tuning `xlm-roberta-base` for 3-class sentiment classification (negative / neutral / positive), evaluating each epoch against the gold-standard set.

In [19]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)
# Metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# Training arguments
training_args = TrainingArguments(
    output_dir='./nepal_sentiment_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir='./logs',
    logging_steps=50,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOAR

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.905328,1.168509,0.426667,0.323852
2,0.731357,0.961297,0.613333,0.598028
3,0.588689,0.817110,0.713333,0.710056


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/samirasharma/Projects/border-sentiment/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/samirasharma/Projects/border-sentiment/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=348, training_loss=0.7583254781262628, metrics={'train_runtime': 1444.4536, 'train_samples_per_second': 3.832, 'train_steps_per_second': 0.241, 'total_flos': 364083191781120.0, 'train_loss': 0.7583254781262628, 'epoch': 3.0})

In [20]:
# Evaluate on clean gold standard
results = trainer.evaluate()
print(results)

# Detailed classification report
predictions = trainer.predict(eval_dataset)
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

from sklearn.metrics import classification_report
print(classification_report(labels, preds, target_names=le.classes_))

/Users/samirasharma/Projects/border-sentiment/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.588689,0.817110,3,0.713333,0.710056


{'eval_loss': 0.8171104192733765, 'eval_accuracy': 0.7133333333333334, 'eval_f1': 0.7100561188498268}


/Users/samirasharma/Projects/border-sentiment/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

    negative       0.62      0.90      0.74       100
     neutral       0.81      0.59      0.68       100
    positive       0.78      0.65      0.71       100

    accuracy                           0.71       300
   macro avg       0.74      0.71      0.71       300
weighted avg       0.74      0.71      0.71       300



## 8. Save Model

In [21]:
trainer.save_model()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Results & Limitations

**Final result:** 71% accuracy and 0.71 F1 score on the gold-standard evaluation set (3-class problem, random baseline ≈ 33%).

**Honest limitations:**
- Translation quality varies significantly for Nepali slang, idioms, and sarcasm
- Single annotator (no inter-annotator agreement score) for the gold-standard labels
- Total dataset size (~2100 comments) is small by NLP fine-tuning standards
- Class imbalance in the auto-labeled training set (skewed negative) likely shaped training dynamics

This project is published as a baseline and a public dataset, with an open invitation for the community to contribute additional verified labels and help push past this ceiling.

